# reviews EDA + feature engineering

this notebook walks through the reviews pipeline — cleaning, structural features, and sentiment scoring. the actual code is in the `reviews/` module. here we just run it step by step and look at the results.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

## 1. raw data

In [ ]:
reviews = pd.read_csv('data/reviews.csv')
print(reviews.shape)
reviews.head()

In [ ]:
per_listing = reviews.groupby('listing_id').size()
print('unique listings:', len(per_listing))
print('avg reviews per listing:', per_listing.mean().round(1))
print('max:', per_listing.max())

In [ ]:
# reviews come in many languages — florence is very international
reviews['comments'].dropna().sample(10, random_state=7).tolist()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
per_listing.clip(upper=250).hist(bins=50, ax=ax, color='#C96B8A', edgecolor='white')
ax.set_xlabel('number of reviews')
ax.set_ylabel('listings')
ax.set_title('reviews per listing (capped at 250)')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## 2. cleaning

the raw comments sometimes have html tags, some are empty after stripping, and some are automated airbnb cancellation messages (not real guest reviews). `clean_reviews.py` handles all of this.

In [ ]:
from reviews.clean_reviews import clean_reviews, clean_comment

# show what html stripping does
html_ex = reviews[reviews['comments'].str.contains('<', na=False)]['comments'].dropna().iloc[0]
print('raw:    ', html_ex[:250])
print()
print('cleaned:', clean_comment(html_ex)[:250])

In [ ]:
reviews_clean = clean_reviews(reviews)
print(f'before: {len(reviews):,}')
print(f'after:  {len(reviews_clean):,}')
print(f'removed: {len(reviews) - len(reviews_clean):,}')

## 3. structural features

no NLP needed here — just counts and dates. how many reviews, how recent, how fast they come in. fast to compute and works for any language.

In [ ]:
from reviews.structural_features import build_structural_features

SNAPSHOT = '2025-09-24'
struct = build_structural_features(reviews_clean, snapshot_date=SNAPSHOT)
print(struct.shape)
struct.head()

In [ ]:
struct.describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))

struct['rev_n_reviews'].clip(upper=300).hist(bins=50, ax=axes[0], color='#C96B8A', edgecolor='white')
axes[0].set_title('total reviews (capped 300)')

struct['rev_per_month'].clip(upper=10).hist(bins=50, ax=axes[1], color='#6B8A40', edgecolor='white')
axes[1].set_title('reviews per month')

struct['rev_days_since_last'].hist(bins=50, ax=axes[2], color='#C96B8A', edgecolor='white')
axes[2].set_title('days since last review')

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

## 4. sentiment analysis

model: `nlptown/bert-base-multilingual-uncased-sentiment` — trained on reviews in EN, NL, DE, FR, ES, IT. outputs an expected star rating (1-5) per review, which we then aggregate per listing.

the demo below only runs on 25 reviews. full run was on google colab (T4 GPU, ~2 hours for 1M reviews).

In [ ]:
from reviews.sentiment_features import compute_review_sentiment, build_sentiment_features

# set sample_n=None to run on everything (needs GPU)
demo = compute_review_sentiment(reviews_clean, sample_n=25, cache_path=None)
demo[['comment_clean', 'sent_stars']].head(10)

In [ ]:
for _, row in demo.dropna(subset=['sent_stars']).head(12).iterrows():
    s = round(row['sent_stars'])
    print(f"[{'★'*s}{'☆'*(5-s)}]  {str(row['comment_clean'])[:100]}")

In [ ]:
# aggregate to listing level
sent_feats_demo = build_sentiment_features(demo)
print(list(sent_feats_demo.columns))
sent_feats_demo.head()

## 5. full output — reviews_features.csv

result of running the full pipeline on all 1M reviews.

In [ ]:
feats = pd.read_csv('data/reviews_features.csv')
print(feats.shape)
feats.head()

In [ ]:
feats.describe().round(3)

In [ ]:
# distributions of sentiment features
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, col in zip(axes, ['rev_sent_mean', 'rev_sent_std', 'rev_sent_min', 'rev_frac_negative']):
    feats[col].dropna().hist(bins=40, ax=ax, color='#C96B8A', edgecolor='white')
    ax.set_title(col, fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## 6. correlation with price

In [ ]:
listings = pd.read_csv('data/listings_features.csv')

price_col = next((c for c in listings.columns if 'price' in c.lower() and 'log' not in c.lower()), None)
if price_col is None:
    price_col = next(c for c in listings.columns if 'price' in c.lower())
print('price column:', price_col)

merged = listings[['id', price_col]].merge(feats, on='id', how='inner')
merged['log_price'] = np.log1p(merged[price_col])
print('shape after merge:', merged.shape)

In [ ]:
rev_cols = [c for c in feats.columns if c != 'id']
corrs = merged[rev_cols + ['log_price']].corr()['log_price'].drop('log_price').sort_values(key=abs)

fig, ax = plt.subplots(figsize=(6, 4))
colors = ['#C96B8A' if v >= 0 else '#6B8A40' for v in corrs]
ax.barh(corrs.index, corrs.values, color=colors, edgecolor='white')
ax.axvline(0, color='gray', linewidth=0.8)
ax.set_xlabel('correlation with log(price)')
ax.set_title('review features vs price')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, col, color in zip(axes,
    ['rev_sent_mean', 'rev_days_since_last'],
    ['#C96B8A', '#6B8A40']):

    tmp = merged[[col, 'log_price']].dropna().sample(min(3000, len(merged)), random_state=1)
    ax.scatter(tmp[col], tmp['log_price'], alpha=0.2, s=7, color=color)
    m, b = np.polyfit(tmp[col], tmp['log_price'], 1)
    xs = np.linspace(tmp[col].min(), tmp[col].max(), 100)
    ax.plot(xs, m*xs+b, color='black', linewidth=1.5)
    r = tmp[[col, 'log_price']].corr().iloc[0, 1]
    ax.set_title(f'{col}  (r={r:.3f})')
    ax.set_xlabel(col)
    ax.set_ylabel('log(price)')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

`rev_sent_mean` and `rev_days_since_last` both have a clear relationship with price — listings with better ratings and more recent reviews tend to be priced higher. in the final model these two features rank #6 and #8 by SHAP importance.